# Zone Watch: train the helmet detector

Fine-tunes a small pretrained YOLO model on the helmet dataset, saves `best.pt` to Google Drive and reports its results.

**Before running**

1. Upload `zone_watch_dataset.zip` to `MyDrive/zone-watch/` in Google Drive.
2. Select **Runtime > Change runtime type > T4 GPU**.
3. Run the cells from top to bottom.

## 1. Check the GPU

Training on a CPU would take hours. If this cell fails, go back to step 2 above.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive and set the paths

Colab deletes its local disk when the session ends, so everything worth keeping goes to Drive.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/zone-watch")
DATASET_ZIP = DRIVE_DIR / "zone_watch_dataset.zip"
RUNS_DIR = DRIVE_DIR / "runs"
DATASET_DIR = Path("/content/dataset")

assert DATASET_ZIP.exists(), f"Upload the dataset zip to {DATASET_ZIP}"

## 3. Install Ultralytics

The version is pinned to match `requirements.txt`, so training and the app use the same code.

In [ ]:
%pip install -q ultralytics==8.4.197

## 4. Unpack the dataset

Images are unpacked to Colab's local disk because reading hundreds of small files from Drive is very slow.

In [ ]:
import shutil

shutil.unpack_archive(DATASET_ZIP, DATASET_DIR)
DATA_YAML = DATASET_DIR / "data.yaml"

for split in ("train", "val", "test"):
    print(f"{split}: {len(list((DATASET_DIR / 'images' / split).iterdir()))} images")

## 5. Train

The pretrained model already knows general visual features, so 524 images are enough to adapt it to helmets.

- `yolo11n.pt` is the nano model. For the small model, use `yolo11s.pt` and change `RUN_NAME`.
- `patience` stops training early if validation results stop improving.
- `seed` makes the run as repeatable as a GPU allows.

Checkpoints are written to Drive during training. **If the session disconnects**, rerun cells 1 to 4, set `RESUME_FROM` to the path of `last.pt`, for example `RUNS_DIR / "yolo11n" / "weights" / "last.pt"`, and run this cell again.

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolo11n.pt"
RUN_NAME = "yolo11n"
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42
RESUME_FROM = None

if RESUME_FROM:
    model = YOLO(RESUME_FROM)
    model.train(resume=True)
else:
    model = YOLO(MODEL_NAME)
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        seed=SEED,
        workers=2,
        project=str(RUNS_DIR),
        name=RUN_NAME,
    )

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print(f"Best weights: {BEST_WEIGHTS}")

## 6. Measure the results

- **Precision**: of the boxes the model drew, how many were right.
- **Recall**: of the real objects, how many it found. For **Without Helmet** this is the key number, because a miss here is a missed alert.
- **mAP@50**: average precision when a box counts as correct at 50% overlap with the real box.
- **mAP@50-95**: the same, averaged over stricter overlaps, so it also rewards tight boxes.

Training already used the **val** split to pick the best epoch, which makes its scores slightly optimistic. The **test** split was never used, so its rows are the ones to report.

In [ ]:
import pandas as pd

best_model = YOLO(BEST_WEIGHTS)


def evaluate(split: str) -> tuple[list[dict], Path]:
    """Run the best model on one split; return a table row per class and the plots folder."""
    metrics = best_model.val(
        data=str(DATA_YAML), split=split, imgsz=IMAGE_SIZE, project=str(RUN_DIR), name=f"eval_{split}"
    )
    box = metrics.box
    rows = [{
        "split": split, "class": "all", "precision": box.mp, "recall": box.mr,
        "mAP@50": box.map50, "mAP@50-95": box.map,
    }]
    for index, class_id in enumerate(box.ap_class_index):
        rows.append({
            "split": split, "class": metrics.names[int(class_id)], "precision": box.p[index],
            "recall": box.r[index], "mAP@50": box.ap50[index], "mAP@50-95": box.ap[index],
        })
    return rows, Path(metrics.save_dir)


val_rows, val_dir = evaluate("val")
test_rows, test_dir = evaluate("test")

results = pd.DataFrame(val_rows + test_rows).round(3)
results.to_csv(RUN_DIR / "metrics.csv", index=False)
results

## 7. Confusion matrix

Each column is what was really there and each row is what the model said. **background** means nothing: a real object predicted as background was missed, and a background box predicted as a class is a false alarm.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(test_dir / "confusion_matrix.png"), width=600))

## 8. Sample predictions

Predictions on images from the test split.

In [ ]:
import matplotlib.pyplot as plt

CONFIDENCE = 0.25
TEST_IMAGES = sorted((DATASET_DIR / "images" / "test").iterdir())


def show_predictions(images: list[Path], columns: int = 3) -> None:
    """Draw the model's boxes on each image and show them in a grid."""
    rows = -(-len(images) // columns)
    _, axes = plt.subplots(rows, columns, figsize=(5 * columns, 4 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, image in zip(axes.flat, images):
        result = best_model.predict(image, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
        axis.imshow(result.plot()[:, :, ::-1])
        axis.set_title(image.name, fontsize=9)
    plt.tight_layout()
    plt.show()


show_predictions(TEST_IMAGES[::len(TEST_IMAGES) // 6][:6])

## 9. Where the model struggles

Ranks test images by how far the predicted box count per class is from the labels. A count is only a rough signal, but it quickly surfaces real mistakes and images with **missing labels**. Note any wrong labels you spot: they can be fixed in CVAT before retraining.

In [ ]:
from collections import Counter


def count_errors(image: Path) -> int:
    """Return the total per-class difference between predicted and labelled box counts."""
    label_path = DATASET_DIR / "labels" / "test" / f"{image.stem}.txt"
    labels = label_path.read_text().splitlines() if label_path.exists() else []
    expected = Counter(int(line.split()[0]) for line in labels)
    result = best_model.predict(image, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
    predicted = Counter(int(class_id) for class_id in result.boxes.cls)
    return sum(abs(expected[i] - predicted[i]) for i in best_model.names)


worst_images = sorted(TEST_IMAGES, key=count_errors, reverse=True)[:6]
show_predictions(worst_images)

## 10. Save the model

Copies the weights to the top of the Drive folder, named after `RUN_NAME`. Download the file, for example `MyDrive/zone-watch/yolo11n.pt`, and put it in the project's `models/` folder.

In [ ]:
SAVED_WEIGHTS = DRIVE_DIR / f"{RUN_NAME}.pt"
shutil.copy2(BEST_WEIGHTS, SAVED_WEIGHTS)
print(f"Saved {SAVED_WEIGHTS}")
results[results["split"] == "test"]